# Results analysis

This notebook loads the raw BFM CSV outputs and reproduces the compact summaries used in the thesis.

In [1]:

from pathlib import Path
import pandas as pd
ROOT = Path('..').resolve()
RAW = ROOT / 'results' / 'raw'
files = [
    'bfm_L0_results.csv',
    'bfm_L1_local_prior_ignorance_results_ranked.csv',
    'bfm_L2_fault_family_prior_ignorance_results.csv',
    'bfm_L4_combined_incompleteness_results.csv',
]
frames = []
for file in files:
    df = pd.read_csv(RAW / file)
    if 'width' not in df.columns:
        df['width'] = df['plausibility'] - df['belief']
    df['source_file'] = file
    frames.append(df)
all_results = pd.concat(frames, ignore_index=True)
all_results.head()


,scenario_id,level,bn_fault_var,bn_fault_state,bfm_var,bfm_fault_state,belief,plausibility,width,empty_set_mass,bfm_rank,source_file,rank_by_belief,rank_by_plausibility
0,7,L0_complete,F_battery,exhausted,Batt,exh,1.00,1.00,0.0,0.9204,1.0,bfm_L0_results.csv,NaN,NaN
1,7,L0_complete,F_sw_1,detached,SW1,det,0.03,0.03,0.0,0.9204,2.0,bfm_L0_results.csv,NaN,NaN
2,7,L0_complete,F_sw_2,detached,SW2,det,0.03,0.03,0.0,0.9204,3.0,bfm_L0_results.csv,NaN,NaN
3,7,L0_complete,F_sw_3,detached,SW3,det,0.03,0.03,0.0,0.9204,4.0,bfm_L0_results.csv,NaN,NaN
4,7,L0_complete,F_sw_4,detached,SW4,det,0.03,0.03,0.0,0.9204,5.0,bfm_L0_results.csv,NaN,NaN


In [2]:

# Conflict table
conflict = all_results.groupby(['scenario_id','level'], as_index=False)['empty_set_mass'].first()
conflict_pivot = conflict.pivot(index='scenario_id', columns='level', values='empty_set_mass')
conflict_pivot


level,L0_complete,L1_local_prior_ignorance,L2_fault_family_prior_ignorance,L4_combined_incompleteness
scenario_id,,,,
7,0.92040,0.00500,0.00000,NaN
8,0.95914,0.17281,0.08460,NaN
11,0.68251,0.08435,0.08435,NaN
14,0.99525,0.00000,0.00000,NaN
15,NaN,NaN,NaN,0.0000
16,NaN,NaN,NaN,0.0846


In [3]:

# Scenario 8 progression
all_results[(all_results.scenario_id == 8) & (all_results.bfm_var.isin(['SW3','C3']))][
    ['level','bfm_var','belief','plausibility','width','empty_set_mass']
].sort_values(['level','bfm_var'])


,level,bfm_var,belief,plausibility,width,empty_set_mass
21,L0_complete,C3,0.40486,0.40486,0.0,0.95914
20,L0_complete,SW3,0.60729,0.60729,0.0,0.95914
119,L1_local_prior_ignorance,C3,0.00000,1.00000,1.0,0.17281
116,L1_local_prior_ignorance,SW3,0.00000,1.00000,1.0,0.17281
187,L2_fault_family_prior_ignorance,C3,0.00000,1.00000,1.0,0.08460
181,L2_fault_family_prior_ignorance,SW3,0.00000,1.00000,1.0,0.08460


In [4]:

# Top candidates for each scenario/level by belief
rank_cols = []
for level, g in all_results.groupby('level'):
    rank_col = 'rank_by_belief' if 'rank_by_belief' in g.columns and g['rank_by_belief'].notna().any() else 'bfm_rank'
    top = g.sort_values(['scenario_id', rank_col]).groupby('scenario_id').head(5)
    display(top[['scenario_id','level','bfm_var','bfm_fault_state','belief','plausibility','width','empty_set_mass']])


,scenario_id,level,bfm_var,bfm_fault_state,belief,plausibility,width,empty_set_mass
0,7,L0_complete,Batt,exh,1.00000,1.00000,0.0,0.92040
1,7,L0_complete,SW1,det,0.03000,0.03000,0.0,0.92040
2,7,L0_complete,SW2,det,0.03000,0.03000,0.0,0.92040
3,7,L0_complete,SW3,det,0.03000,0.03000,0.0,0.92040
4,7,L0_complete,SW4,det,0.03000,0.03000,0.0,0.92040
20,8,L0_complete,SW3,det,0.60729,0.60729,0.0,0.95914
21,8,L0_complete,C3,bro,0.40486,0.40486,0.0,0.95914
22,8,L0_complete,SW4,det,0.03000,0.03000,0.0,0.95914
23,8,L0_complete,SW5,det,0.03000,0.03000,0.0,0.95914
24,8,L0_complete,SW6,det,0.03000,0.03000,0.0,0.95914


,scenario_id,level,bfm_var,bfm_fault_state,belief,plausibility,width,empty_set_mass
80,7,L1_local_prior_ignorance,Batt,exh,1.00,1.00,0.0,0.00500
81,7,L1_local_prior_ignorance,SW1,det,0.03,0.03,0.0,0.00500
82,7,L1_local_prior_ignorance,SW2,det,0.03,0.03,0.0,0.00500
83,7,L1_local_prior_ignorance,SW3,det,0.03,0.03,0.0,0.00500
84,7,L1_local_prior_ignorance,SW4,det,0.03,0.03,0.0,0.00500
100,8,L1_local_prior_ignorance,SW4,det,0.03,0.03,0.0,0.17281
101,8,L1_local_prior_ignorance,SW5,det,0.03,0.03,0.0,0.17281
102,8,L1_local_prior_ignorance,SW6,det,0.03,0.03,0.0,0.17281
103,8,L1_local_prior_ignorance,SW7,det,0.03,0.03,0.0,0.17281
104,8,L1_local_prior_ignorance,SW8,det,0.03,0.03,0.0,0.17281


,scenario_id,level,bfm_var,bfm_fault_state,belief,plausibility,width,empty_set_mass
160,7,L2_fault_family_prior_ignorance,Batt,exh,1.00000,1.00000,0.0,0.00000
161,7,L2_fault_family_prior_ignorance,SW1,det,0.03000,0.03000,0.0,0.00000
162,7,L2_fault_family_prior_ignorance,SW2,det,0.03000,0.03000,0.0,0.00000
163,7,L2_fault_family_prior_ignorance,SW3,det,0.03000,0.03000,0.0,0.00000
164,7,L2_fault_family_prior_ignorance,SW4,det,0.03000,0.03000,0.0,0.00000
180,8,L2_fault_family_prior_ignorance,LF,bro,0.01500,0.01500,0.0,0.08460
181,8,L2_fault_family_prior_ignorance,SW3,det,0.00000,1.00000,1.0,0.08460
182,8,L2_fault_family_prior_ignorance,SW4,det,0.00000,1.00000,1.0,0.08460
183,8,L2_fault_family_prior_ignorance,SW5,det,0.00000,1.00000,1.0,0.08460
184,8,L2_fault_family_prior_ignorance,SW6,det,0.00000,1.00000,1.0,0.08460


,scenario_id,level,bfm_var,bfm_fault_state,belief,plausibility,width,empty_set_mass
240,15,L4_combined_incompleteness,Batt,exh,1.000,1.000,0.0,0.0000
241,15,L4_combined_incompleteness,SW1,det,0.030,0.030,0.0,0.0000
242,15,L4_combined_incompleteness,SW2,det,0.030,0.030,0.0,0.0000
243,15,L4_combined_incompleteness,SW3,det,0.030,0.030,0.0,0.0000
244,15,L4_combined_incompleteness,SW4,det,0.030,0.030,0.0,0.0000
260,16,L4_combined_incompleteness,LF,bro,0.015,0.015,0.0,0.0846
261,16,L4_combined_incompleteness,SW3,det,0.000,1.000,1.0,0.0846
262,16,L4_combined_incompleteness,SW4,det,0.000,1.000,1.0,0.0846
263,16,L4_combined_incompleteness,SW5,det,0.000,1.000,1.0,0.0846
264,16,L4_combined_incompleteness,SW6,det,0.000,1.000,1.0,0.0846
